In [51]:
import pandas as pd
import numpy as np
import uuid
import json
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams
from openai import OpenAI
import os


def get_embedding(text, openai_client, model="text-embedding-3-small"):
    """Get embedding for text using OpenAI."""
    try:
        response = openai_client.embeddings.create(
            input=text,
            model=model
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return None

def create_collection_if_not_exists(client, collection_name, vector_size=1536):
    """Create Qdrant collection if it doesn't exist."""
    try:
        collections = client.get_collections()
        collection_names = [col.name for col in collections.collections]
        
        if collection_name not in collection_names:
            client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(
                    size=vector_size,
                    distance=Distance.COSINE
                )
            )
            print(f"Created collection: {collection_name}")
        else:
            print(f"Collection {collection_name} already exists")
    except Exception as e:
        print(f"Error creating collection: {e}")

def push_chunks(collection_name,chunks_data, batch_size=100):
    """
    Push chunk data to Qdrant with metadata.
    
    Args:
        collection_name (str): Name of the Qdrant collection
        chunks_data (pd.DataFrame): Chunk data
        batch_size (int): Number of points to upsert in each batch
    """
    
    # Initialize clients
    client = QdrantClient(url="http://dev.platform.farmer.chat:5438/", port=5438, grpc_port=5439, prefer_grpc=False)
    
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable not set")
    openai_client = OpenAI(api_key=api_key)
    
    # Create collection if it doesn't exist
    create_collection_if_not_exists(client, collection_name)
    
    # Combine both datasets
    all_chunks = []
    
    
    # Process Hindi chunks
    for idx, row in chunks_data.iterrows():
        tags = json.loads(row['tags'])
        if pd.notna(row['chunk']):  # Check if chunk is not null
            all_chunks.append({
                'chunk_text': row['chunk'],
                'chunk_id': row['chunk_id'],
                'topic': tags['tags'],
                'language': tags['language'],
                'region': tags['region'],
                'crop': tags['crop'],
                'file_name': row['file_name']
            })
    
    print(f"Total chunks to process: {len(all_chunks)}")
    
    # Process chunks in batches
    points = []
    processed_count = 0
    
    for chunk_data in all_chunks:
        try:
            # Get embedding for the chunk text
            embedding = get_embedding(chunk_data['chunk_text'], openai_client)
            
            if embedding is None:
                print(f"Skipping chunk {chunk_data['chunk_id']} due to embedding error")
                continue
            
            # Create point structure
            point = PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload={
                    "chunk_text": chunk_data['chunk_text'],
                    "chunk_id": chunk_data['chunk_id'],
                    "topic": chunk_data['topic'],
                    "language": chunk_data['language'],
                    "region": chunk_data['region'],
                    "crop": chunk_data['crop'],
                    "file_name": chunk_data['file_name']
                }
            )
            points.append(point)
            processed_count += 1
            
            # Upsert in batches
            if len(points) >= batch_size:
                client.upsert(
                    collection_name=collection_name,
                    points=points
                )
                print(f"Upserted batch of {len(points)} points. Total processed: {processed_count}")
                points = []
                
        except Exception as e:
            print(f"Error processing chunk {chunk_data['chunk_id']}: {e}")
            continue
    
    # Upsert remaining points
    if points:
        client.upsert(
            collection_name=collection_name,
            points=points
        )
        print(f"Upserted final batch of {len(points)} points")
    
    print(f"Successfully processed {processed_count} chunks")
    
    # Get collection info
    collection_info = client.get_collection(collection_name)
    print(f"Collection {collection_name} now contains {collection_info.points_count} points")

def main():
    # Load the chunk data
    print("Loading chunk data...")
    english_chunks = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/English_chunks - tags.csv")
    hindi_chunks = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/hindi_chunks - tags.csv")

    
    
    # Test with a small sample first
    print("\nTesting with a small sample...")
    english_sample = english_chunks.head(10)
    hindi_sample = hindi_chunks.head(5)
    
    # Push test data
    push_chunks("test_agriculture_chunks", english_sample, batch_size=10)
    
    # Push all data to production collection
    # print("\nPushing all chunks to production collection...")
    # push_chunks("agriculture_chunks", english_chunks, hindi_chunks, batch_size=100)

if __name__ == "__main__":
    main()

Loading chunk data...



Testing with a small sample...
Created collection: test_agriculture_chunks
Total chunks to process: 10
Upserted batch of 10 points. Total processed: 10
Successfully processed 10 chunks
Collection test_agriculture_chunks now contains 10 points


In [1]:
import pandas as pd
import numpy as np
import uuid
import json
import pickle
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams
from openai import OpenAI
import time
from tqdm import tqdm


def get_embedding(text, openai_client, model="text-embedding-3-small"):
    """Get embedding for text using OpenAI."""
    try:
        response = openai_client.embeddings.create(
            input=text,
            model=model
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return None

def create_collection_if_not_exists(client, collection_name, vector_size=1536):
    """Create Qdrant collection if it doesn't exist."""
    try:
        collections = client.get_collections()
        collection_names = [col.name for col in collections.collections]
        
        if collection_name not in collection_names:
            client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(
                    size=vector_size,
                    distance=Distance.COSINE
                )
            )
            print(f"Created collection: {collection_name}")
        else:
            print(f"Collection {collection_name} already exists - will append to existing data")
    except Exception as e:
        print(f"Error creating collection: {e}")

def save_embeddings_to_local(embeddings_data, filename):
    """Save embeddings data to local file."""
    try:
        with open(filename, 'wb') as f:
            pickle.dump(embeddings_data, f)
        print(f"Embeddings saved to: {filename}")
    except Exception as e:
        print(f"Error saving embeddings: {e}")

def load_embeddings_from_local(filename):
    """Load embeddings data from local file."""
    try:
        if os.path.exists(filename):
            with open(filename, 'rb') as f:
                embeddings_data = pickle.load(f)
            print(f"Embeddings loaded from: {filename}")
            return embeddings_data
        else:
            print("No embeddings file found")
            return None
    except Exception as e:
        print(f"Error loading embeddings: {e}")
        return None

def save_checkpoint(checkpoint_data, checkpoint_file):
    """Save checkpoint data to file."""
    try:
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        print(f"Checkpoint saved: {checkpoint_file}")
    except Exception as e:
        print(f"Error saving checkpoint: {e}")

def load_checkpoint(checkpoint_file):
    """Load checkpoint data from file."""
    try:
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'rb') as f:
                checkpoint_data = pickle.load(f)
            print(f"Checkpoint loaded: {checkpoint_file}")
            return checkpoint_data
        else:
            print("No checkpoint file found")
            return None
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        return None

def extract_json_from_text(text):
    """Extract JSON content from text that may contain markdown code blocks."""
    if pd.isna(text) or text == '' or text.strip() == '':
        return None
    
    # Remove markdown code block markers
    # Pattern to match ```json ... ``` or ``` ... ```
    json_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    
    # Try to find JSON within code blocks first
    match = re.search(json_pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # If no code block found, try to find JSON directly
    # Look for content that starts with { and ends with }
    json_direct_pattern = r'\{.*?\}'
    match = re.search(json_direct_pattern, text, re.DOTALL)
    if match:
        return match.group(0).strip()
    
    return None

def parse_tags_safely(tags_str):
    """Safely parse tags JSON with fallback to default values."""
    if pd.isna(tags_str) or tags_str == '' or tags_str.strip() == '':
        return {
            'tags': 'unknown',
            'language': 'unknown',
            'region': 'unknown',
            'crop': 'unknown'
        }
    
    # Extract JSON from the text
    json_content = extract_json_from_text(tags_str)
    
    if json_content is None:
        print(f"No valid JSON found in tags: '{tags_str[:100]}...' - using default values")
        return {
            'tags': 'unknown',
            'language': 'unknown',
            'region': 'unknown',
            'crop': 'unknown'
        }
    
    try:
        parsed_tags = json.loads(json_content)
        
        # Handle different tag formats
        if isinstance(parsed_tags, dict):
            return {
                'tags': parsed_tags.get('tags', 'unknown'),
                'language': parsed_tags.get('language', 'unknown'),
                'region': parsed_tags.get('region', 'unknown'),
                'crop': parsed_tags.get('crop', 'unknown')
            }
        else:
            return {
                'tags': 'unknown',
                'language': 'unknown',
                'region': 'unknown',
                'crop': 'unknown'
            }
            
    except json.JSONDecodeError:
        print(f"Invalid JSON in tags: '{json_content[:100]}...' - using default values")
        return {
            'tags': 'unknown',
            'language': 'unknown',
            'region': 'unknown',
            'crop': 'unknown'
        }
    except Exception as e:
        print(f"Error parsing tags '{json_content[:100]}...': {e} - using default values")
        return {
            'tags': 'unknown',
            'language': 'unknown',
            'region': 'unknown',
            'crop': 'unknown'
        }

def process_chunk_batch_embeddings(chunk_batch, openai_client):
    """Process a batch of chunks and return embeddings data."""
    embeddings_data = []
    failed_chunks = []
    
    for chunk_data in chunk_batch:
        try:
            # Get embedding for the chunk text
            embedding = get_embedding(chunk_data['chunk_text'], openai_client)
            
            if embedding is None:
                failed_chunks.append(chunk_data['chunk_id'])
                continue
            
            # Store embedding data locally
            embeddings_data.append({
                'id': str(uuid.uuid4()),
                'vector': embedding,
                'payload': {
                    "chunk_text": chunk_data['chunk_text'],
                    "chunk_id": chunk_data['chunk_id'],
                    "topic": chunk_data['topic'],
                    "language": chunk_data['language'],
                    "region": chunk_data['region'],
                    "crop": chunk_data['crop'],
                    "file_name": chunk_data['file_name']
                }
            })
            
        except Exception as e:
            print(f"Error processing chunk {chunk_data['chunk_id']}: {e}")
            failed_chunks.append(chunk_data['chunk_id'])
            continue
    
    return embeddings_data, failed_chunks

def create_embeddings_locally(collection_name, chunks_data, batch_size=100, max_workers=4, checkpoint_interval=500):
    """
    Create embeddings locally and save them to disk.
    
    Args:
        collection_name (str): Name of the collection (for file naming)
        chunks_data (pd.DataFrame): Chunk data
        batch_size (int): Number of chunks to process in each batch
        max_workers (int): Number of parallel workers
        checkpoint_interval (int): Save checkpoint every N chunks
    """
    
    # Initialize OpenAI client
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable not set")
    openai_client = OpenAI(api_key=api_key)
    
    # File names
    embeddings_file = f"embeddings_{collection_name}.pkl"
    checkpoint_file = f"checkpoint_{collection_name}.pkl"
    
    # Load checkpoint if exists
    checkpoint_data = load_checkpoint(checkpoint_file)
    if checkpoint_data:
        processed_chunk_ids = set(checkpoint_data.get('processed_chunk_ids', []))
        total_processed = checkpoint_data.get('total_processed', 0)
        print(f"Resuming from checkpoint: {total_processed} chunks already processed")
    else:
        processed_chunk_ids = set()
        total_processed = 0
    
    # Prepare chunks data with better error handling
    all_chunks = []
    skipped_rows = 0
    
    for idx, row in chunks_data.iterrows():
        if pd.notna(row['chunk']) and row['chunk_id'] not in processed_chunk_ids:
            try:
                # Safely parse tags
                tags = parse_tags_safely(row['tags'])
                
                all_chunks.append({
                    'chunk_text': row['chunk'],
                    'chunk_id': row['chunk_id'],
                    'topic': tags['tags'],
                    'language': tags['language'],
                    'region': tags['region'],
                    'crop': tags['crop'],
                    'file_name': row['file_name']
                })
            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                skipped_rows += 1
                continue
    
    print(f"Total chunks to process: {len(all_chunks)}")
    print(f"Skipped rows due to errors: {skipped_rows}")
    
    # Load existing embeddings if any
    all_embeddings = load_embeddings_from_local(embeddings_file) or []
    failed_chunks = []
    
    # Split chunks into batches for parallel processing
    chunk_batches = [all_chunks[i:i + batch_size] for i in range(0, len(all_chunks), batch_size)]
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all batches
        future_to_batch = {
            executor.submit(process_chunk_batch_embeddings, batch, openai_client): batch 
            for batch in chunk_batches
        }
        
        # Process completed batches with progress bar
        with tqdm(total=len(chunk_batches), desc="Creating embeddings") as pbar:
            for future in as_completed(future_to_batch):
                try:
                    embeddings_batch, failed = future.result()
                    all_embeddings.extend(embeddings_batch)
                    failed_chunks.extend(failed)
                    
                    # Update progress
                    total_processed += len(embeddings_batch)
                    processed_chunk_ids.update([emb['payload']['chunk_id'] for emb in embeddings_batch])
                    
                    # Save embeddings and checkpoint periodically
                    if total_processed % checkpoint_interval == 0:
                        save_embeddings_to_local(all_embeddings, embeddings_file)
                        checkpoint_data = {
                            'processed_chunk_ids': list(processed_chunk_ids),
                            'total_processed': total_processed,
                            'collection_name': collection_name
                        }
                        save_checkpoint(checkpoint_data, checkpoint_file)
                    
                    pbar.update(1)
                    pbar.set_postfix({'Processed': total_processed, 'Failed': len(failed_chunks)})
                    
                except Exception as e:
                    print(f"Error in batch processing: {e}")
                    pbar.update(1)
    
    # Final save
    save_embeddings_to_local(all_embeddings, embeddings_file)
    checkpoint_data = {
        'processed_chunk_ids': list(processed_chunk_ids),
        'total_processed': total_processed,
        'collection_name': collection_name
    }
    save_checkpoint(checkpoint_data, checkpoint_file)
    
    print(f"Successfully created embeddings for {total_processed} chunks")
    print(f"Failed chunks: {len(failed_chunks)}")
    print(f"Skipped rows: {skipped_rows}")
    print(f"Embeddings saved to: {embeddings_file}")
    
    return embeddings_file

def upload_embeddings_to_qdrant(collection_name, embeddings_file=None):
    """
    Upload pre-created embeddings to Qdrant.
    
    Args:
        collection_name (str): Name of the Qdrant collection
        embeddings_file (str): Path to embeddings file (optional, will auto-detect)
    """
    
    # Initialize Qdrant client
    client = QdrantClient(url="http://dev.platform.farmer.chat:5438/", port=5438, grpc_port=5439, prefer_grpc=False)
    
    # Auto-detect embeddings file if not provided
    if embeddings_file is None:
        embeddings_file = f"embeddings_{collection_name}.pkl"
    
    # Load embeddings
    embeddings_data = load_embeddings_from_local(embeddings_file)
    if not embeddings_data:
        print(f"No embeddings found in {embeddings_file}")
        return
    
    print(f"Loading {len(embeddings_data)} embeddings for upload...")
    
    # Create collection if it doesn't exist
    create_collection_if_not_exists(client, collection_name)
    
    # Convert to Qdrant points
    points = []
    for emb_data in embeddings_data:
        point = PointStruct(
            id=emb_data['id'],
            vector=emb_data['vector'],
            payload=emb_data['payload']
        )
        points.append(point)
    
    # Upload to Qdrant in batches
    print(f"Uploading {len(points)} points to Qdrant...")
    batch_size = 100
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        try:
            client.upsert(
                collection_name=collection_name,
                points=batch
            )
            print(f"Uploaded batch {i//batch_size + 1}/{(len(points) + batch_size - 1)//batch_size}")
        except Exception as e:
            print(f"Error uploading batch: {e}")
    
    # Get collection info
    collection_info = client.get_collection(collection_name)
    print(f"Collection {collection_name} now contains {collection_info.points_count} points")

def push_chunks_parallel(collection_name, chunks_data, batch_size=100, max_workers=4, checkpoint_interval=500):
    """
    Complete workflow: Create embeddings locally, then upload to Qdrant.
    """
    print("Step 1: Creating embeddings locally...")
    embeddings_file = create_embeddings_locally(collection_name, chunks_data, batch_size, max_workers, checkpoint_interval)
    
    print("\nStep 2: Uploading embeddings to Qdrant...")
    upload_embeddings_to_qdrant(collection_name, embeddings_file)
    
    print("\nProcess completed successfully!")

def push_chunks(collection_name, chunks_data, batch_size=100):
    """
    Legacy function for backward compatibility
    """
    return push_chunks_parallel(collection_name, chunks_data, batch_size)

def main():
    # Load the chunk data
    print("Loading chunk data...")
    english_chunks = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/English_chunks - tags.csv")
    hindi_chunks = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/hindi_chunks - tags.csv")
    
    print(f"English chunks: {len(english_chunks)}")
    print(f"Hindi chunks: {len(hindi_chunks)}")
    
    # Display data info
    print("\nEnglish chunks info:")
    english_chunks.info()
    print("\nHindi chunks info:")
    hindi_chunks.info()
    
    # Test with a small sample first
    print("\nTesting with a small sample...")
    english_sample = english_chunks.head(10)
    hindi_sample = hindi_chunks.head(10)
    
    # Option 1: Complete workflow (create embeddings + upload)
    # push_chunks_parallel("test_agriculture_chunks", hindi_sample, batch_size=5, max_workers=2, checkpoint_interval=10)
    
    # Option 2: Only create embeddings locally (for later upload)
    create_embeddings_locally("test_agriculture_chunks", hindi_chunks, batch_size=5, max_workers=5, checkpoint_interval=10)
    
    # Option 3: Only upload existing embeddings
    # upload_embeddings_to_qdrant("test_agriculture_chunks")
    
    # Push all data to production collection
    # print("\nPushing all chunks to production collection...")
    # push_chunks_parallel("agriculture_chunks", english_chunks, batch_size=100, max_workers=4, checkpoint_interval=500)

if __name__ == "__main__":
    main()

Loading chunk data...
English chunks: 12576
Hindi chunks: 3371

English chunks info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12576 entries, 0 to 12575
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unnamed: 0.1   12576 non-null  int64 
 1   Unnamed: 0     12576 non-null  int64 
 2   chunk          12575 non-null  object
 3   file_path      12576 non-null  object
 4   file_name      12576 non-null  object
 5   relative_path  12576 non-null  object
 6   chunk_number   12576 non-null  object
 7   tags           12576 non-null  object
 8   chunk_id       12576 non-null  object
dtypes: int64(2), object(7)
memory usage: 884.4+ KB

Hindi chunks info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3371 entries, 0 to 3370
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unnamed: 0.1   3371 non-null   int64 
 1   Unnamed: 0     3371 no

Creating embeddings: 100%|██████████| 674/674 [07:08<00:00,  1.57it/s, Processed=15942, Failed=0]


Embeddings saved to: embeddings_test_agriculture_chunks.pkl
Checkpoint saved: checkpoint_test_agriculture_chunks.pkl
Successfully created embeddings for 15942 chunks
Failed chunks: 0
Skipped rows: 0
Embeddings saved to: embeddings_test_agriculture_chunks.pkl


In [4]:
upload_embeddings_to_qdrant("test_agriculture_chunks")

Embeddings loaded from: embeddings_test_agriculture_chunks.pkl
Loading 15942 embeddings for upload...
Created collection: test_agriculture_chunks
Uploading 15942 points to Qdrant...
Uploaded batch 1/160
Uploaded batch 2/160
Uploaded batch 3/160
Uploaded batch 4/160
Uploaded batch 5/160
Uploaded batch 6/160
Uploaded batch 7/160
Uploaded batch 8/160
Uploaded batch 9/160
Uploaded batch 10/160
Uploaded batch 11/160
Uploaded batch 12/160
Uploaded batch 13/160
Uploaded batch 14/160
Uploaded batch 15/160
Uploaded batch 16/160
Uploaded batch 17/160
Uploaded batch 18/160
Uploaded batch 19/160
Uploaded batch 20/160
Uploaded batch 21/160
Uploaded batch 22/160
Uploaded batch 23/160
Uploaded batch 24/160
Uploaded batch 25/160
Uploaded batch 26/160
Uploaded batch 27/160
Uploaded batch 28/160
Uploaded batch 29/160
Uploaded batch 30/160
Uploaded batch 31/160
Uploaded batch 32/160
Uploaded batch 33/160
Uploaded batch 34/160
Uploaded batch 35/160
Uploaded batch 36/160
Uploaded batch 37/160
Uploaded batc

In [2]:
import pickle

embeddings_pkl_path = "/Users/ganeshnallagachu/Desktop/world_bank/embeddings_test_agriculture_chunks.pkl"

with open(embeddings_pkl_path, "rb") as f:
    embeddings_data = pickle.load(f)

print(f"Loaded embeddings from {embeddings_pkl_path}")
print(f"Type: {type(embeddings_data)}")
if isinstance(embeddings_data, dict):
    print(f"Keys: {list(embeddings_data.keys())[:10]}")
elif isinstance(embeddings_data, list):
    print(f"First item: {embeddings_data[0]}")
else:
    print("Loaded object is not a dict or list.")


Loaded embeddings from /Users/ganeshnallagachu/Desktop/world_bank/embeddings_test_agriculture_chunks.pkl
Type: <class 'list'>
First item: {'id': '20beece0-a880-4b21-b3f3-bbb1d9475a0a', 'vector': [0.03894302621483803, -0.013759697787463665, 0.04099671170115471, 0.007938780821859837, -0.03267928212881088, -0.042691003531217575, 0.059762269258499146, -0.002648934256285429, -0.008465037681162357, -0.014273119159042835, 0.03665829822421074, -0.011250349693000317, 0.00040311613702215254, -0.0433327816426754, -0.03278196603059769, -0.059505559504032135, 0.04307606816291809, -0.02087058685719967, -0.03111334703862667, 0.010537977330386639, 0.03460461273789406, -0.006465903017669916, 0.02266756258904934, -0.01861153170466423, 0.011821531690657139, -0.040354933589696884, 0.012797032482922077, 0.0007063557859510183, 0.006745075806975365, -0.047517165541648865, -0.003229742404073477, -0.024926617741584778, -0.01225152239203453, -0.003260226920247078, -0.06222669407725334, -0.01811094582080841, 0.0

In [3]:
len(embeddings_data)

15942

In [71]:
from pprint import pprint

pprint(embeddings_data[1741])

{'id': 'e658afab-965e-4b0d-895d-8b15e094e508',
 'payload': {'chunk_id': '18 IJAgriSci AUGUST-2024.pdf_7',
             'chunk_text': 'Figures in parenthesis indicate percent decrease '
                           'with respect to control. From the above study, it '
                           'can be presumed that different isolates namely B4, '
                           'B6, A1 and F1 and standard bacterial cultures '
                           'Delftia spp. PP4_S3, Pseudomonas spp. and Bacillus '
                           'spp. are efficient lignocellulose degraders. These '
                           'findings are very much in line with earlier '
                           'reports where higher yield of lignocellulosic '
                           'enzymes has been observed by action of consortia '
                           'than individual cultures (Sahil et al. 2023). In '
                           'the present study, use of enzyme cocktail has '
                           'prov